In [ ]:
# RTB_BOOTSTRAP_CELL -- sets up the environment (Colab / JupyterLite / local install); click to expand. Generated from docs/notebooks/_rtb_nb_bootstrap.py by sync_bootstrap.py -- do not hand-edit.

"""Environment bootstrap for roboticstoolbox-python's Jupyter notebooks.

Installs the toolbox (and reports how) across the three environments a notebook in
this folder might run in: a local Jupyter/VS Code install, Google Colab, and
JupyterLite (Pyodide/WASM, in-browser).

This file is the single source of truth for that logic. Every notebook's own
bootstrap cell is a generated copy of this file's content, produced by
sync_bootstrap.py -- see docs/notebooks/README.md for the full explanation.
"""

import subprocess
import sys
from pathlib import Path


async def ensure_installed(extras: tuple[str, ...] | list[str] = ()) -> bool:
    """Install roboticstoolbox-python if needed, and report the environment.

    :param extras: toolbox extras the notebook needs, for example ``["ik"]``.  They
        are set per notebook in its metadata (``rtb_bootstrap.extras``) and
        passed here by the generated cell.
    :returns: True if running on Google Colab, False otherwise.
    """
    if sys.platform == "emscripten":
        import micropip

        # spatialmath-python <= 1.1.18 lists pre-commit (a dev tool) as a runtime
        # dependency; its virtualenv requires packaging>=26.3, which conflicts with
        # the packaging 24.2 built into Pyodide, so a normal resolve fails.  Install
        # it without dependencies first: it is then already satisfied when the
        # toolbox's own dependencies are resolved below.  Remove this line once a
        # spatialmath release without that dependency is out
        # (rai-opensource/spatialmath-python#250).
        await micropip.install("spatialmath-python", deps=False)
        await micropip.install("ipympl")
        if "ik" in extras:
            # SymPy is one of Pyodide's own packages.  python-flint is not installed:
            # the one bundled with Pyodide 0.27 is too old for the toolbox, which
            # then uses SymPy for the polynomial steps, about 2x slower for a
            # general 6R arm (10 s to derive a Kinova Jaco).
            await micropip.install("sympy")

        wheels = sorted(Path("/pypi").glob("roboticstoolbox_python-*.whl"))
        if wheels:
            # Prefer the wheel bundled with this JupyterLite site. Relative,
            # not absolute: a leading slash resolves against the origin, not
            # this site's own base URL (a GitHub Pages project subpath).
            await micropip.install(f"pypi/{wheels[-1].name}")
        else:
            # Fall back to PyPI when running outside the published site layout.
            await micropip.install("roboticstoolbox-python")
        where, colab = "in browser", False
    else:
        try:
            import google.colab  # noqa: F401
        except ImportError:
            where, colab = "locally", False
            if "ik" in extras:
                try:
                    import sympy  # noqa: F401
                except ImportError:
                    print(
                        "This notebook needs SymPy: "
                        "pip install 'roboticstoolbox-python[ik]'"
                    )
        else:
            print("Installing roboticstoolbox-python...")
            requirement = "roboticstoolbox-python"
            if extras:
                requirement += "[" + ",".join(extras) + "]"
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", requirement],
                check=True,
            )
            where, colab = "on Colab", True

    import roboticstoolbox

    version = getattr(roboticstoolbox, "__version__", "unknown")
    print(f"Running {where} using RTB v{version}")
    return colab

COLAB = await ensure_installed()


In [ ]:
import roboticstoolbox as rtb
from IPython.core.display import SVG

We will load a model for a simple Y-shaped planar robot.  Check out the source code to see how the model was constructed using `ELink` and `ERobot` classes.

In [ ]:
robot = rtb.models.ETS.Planar_Y()
print(robot)

From the displayed table we see that this robot has two end-effectors, indicated by the links shown in blue and prefixed with the @ symbol.

We can easily create a graphical representation of the  turn the robot description into a [GraphViz dotfile](https://graphviz.org)


In [ ]:
robot.dotfile("planar_y.dot")

which writes a file in the current folder.  We then render it to SVG format and [load and display it](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html?highlight=svg#IPython.display.SVG)

In [ ]:
!dot -Tsvg planar_y.dot -x > planar_y.svg
SVG(filename="planar_y.svg")

The fixed base of the robot is shown as the dark grey node.  Link reference frames are shown by circles and the edges are marked with the elementary transform sequences that define the relative poses of the link frames.  End-effector nodes are marked by a blue double circle.

The ETS transforms in black indicate the joints, and they contain the five joint angles `q0` to `q5`.  The assignement of joint numbers is arbitrary and decided by the creator of the robot model.  All operations such as forward kinematics or Jacobians that accept a joint angle vector, organize it as `[q0, q1, q2, q3, q4 q5]`.

Fixed joints  are shown in blue. 

The forward kinematics is quite straightforward, but we must specify which end-effector we are interested in

In [ ]:
robot.fkine(robot.qz, endlink="eea")

which is exactly the same as the pose of the second branch

In [ ]:
robot.fkine(robot.qz, endlink="eeb")

In a similar fashion we can also compute the Jacobian, in this case as a $6 \times 4$ matrix

In [ ]:
robot.jacob0(robot.qz, endlink="eea")

We can generate an ETS for the bottom-branch of the robot

In [ ]:
p = robot.ets(end="eeb")
p

and we could easily turn this into an unbranched robot represented by an `ERobot` instance

In [ ]:
r = rtb.ERobot(p, checkjindex=False)
print(r)

This robot has 5 links but only 4 joint angles which are inherited from the original branched robot: `q0`, `q1`, `q4` and `q5`.  If invoking its `fkine` method you must pass in a vector with at least 6 elements, but only 4 elements will be used.

We can also find the transformation between any two links in the robot.  In this case, from one end effector to the other, from link eea to link eeb

In [ ]:
p = robot.ets(start="eea", end="eeb")
p

which is expressed as an ETS.  Note that it is not in minimal form and the terms in the middle `tx(-1) * tx(1)` cancel out.  We can precompile the constants which will take care of this for us

In [ ]:
p.compile()

# Graph display variations

This robot is entirely revolute, but we can visualize the joint types in the graphical representation by using the `jtype` keyword argument

In [ ]:
robot.dotfile("planar_y.dot", jtype=True)
!dot -Tsvg planar_y.dot > planar_y.svg
SVG(filename="planar_y.svg")

and the input to a link frame is a black dot if it is preceded by a revolute joint, and a black box if it  preceded by a prismatic joint.

We can also highlight the ETS by placing them in boxes, rather than being just edge labels, using the `etsbox` option

In [ ]:
robot.dotfile("planar_y.dot", etsbox=True)
!dot -Tsvg planar_y.dot > planar_y.svg
SVG(filename="planar_y.svg")

## PR2 model

The PR2 is a very complex branched robot with nearly 90 links

In [ ]:
pr2 = rtb.models.URDF.PR2()
pr2.dotfile("pr2.dot")
!dot -Tsvg pr2.dot > pr2.svg
SVG(filename="pr2.svg")